In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

In [ ]:

# 設定資料夾
DATA_FOLDER = './Stiffness/Y3000/ShearFace'
AVERAGE_FOLDER = 'Average'
HEATMAP_FOLDER = 'Heatmap'
MIDLINE_FOLDER = 'Midline'

# 創建輸出資料夾
for folder in [AVERAGE_FOLDER, HEATMAP_FOLDER, MIDLINE_FOLDER]:
    os.makedirs(folder, exist_ok=True)

# 定義破裂位置
RUPTURE_POSITIONS = [115, 110, 105, 100, 95, 90, 85, 80, 75, 65, 55, 45, 35, 25, 15, 5]

In [3]:
for RUPTURE_POSITION in RUPTURE_POSITIONS:
    
    # 讀取 NPZ 檔案
    npz_file = os.path.join(DATA_FOLDER, f'ShearFace-{RUPTURE_POSITION}.npz')
    
    if not os.path.exists(npz_file):
        print(f"File {npz_file} not found, skipping...")
        continue
    
    print(f"\nProcessing {npz_file}...")
    
    # 載入數據
    data = np.load(npz_file)
    
    # 提取座標和應力數據
    x = data['x']
    y = data['y']
    z = data['z']
    s11 = data['s11']
    s12 = data['s12']
    mu = data['mu']
    
    # === 1. Heatmap 繪圖 (Y-Z 投影) ===
    
    # S12 heatmap
    plt.figure(figsize=(8, 4))
    sc = plt.scatter(y, z, c=s12, cmap='RdBu_r', edgecolors='k', linewidths=0.3)
    plt.colorbar(sc, label='S12 (MPa)')
    plt.xlabel('Y Position')
    plt.ylabel('Z Position')
    plt.title(f'Shear Stress S12 on RIGHT-LEFT Face (YZ Projection) - Position {RUPTURE_POSITION}')
    plt.axis('equal')
    plt.grid(True)
    plt.tight_layout()
    # plt.savefig(f'./{HEATMAP_FOLDER}/Shear-Stress-{RUPTURE_POSITION}-Heatmap-S12.pdf', dpi=300)
    plt.show()
    plt.close()
    
    # μ heatmap
    plt.figure(figsize=(8, 4))
    sc = plt.scatter(y, z, c=mu, cmap='RdBu_r', edgecolors='k', linewidths=0.3)
    plt.colorbar(sc, label=r'$\mu = \frac{-S12}{S11}$')
    plt.xlabel('Y Position')
    plt.ylabel('Z Position')
    plt.title(f'Friction Coefficient μ on RIGHT-LEFT Face (YZ Projection) - Position {RUPTURE_POSITION}')
    plt.axis('equal')
    plt.grid(True)
    plt.tight_layout()
    # plt.savefig(f'./{HEATMAP_FOLDER}/Shear-Stress-{RUPTURE_POSITION}-Heatmap-Mu.pdf', dpi=300)
    # plt.show()
    plt.close()
    
    # === 2. Y 方向平均值計算和繪圖 ===
    
    num_bins = 100
    y_min, y_max = y.min(), y.max()
    bin_edges = np.linspace(y_min, y_max, num_bins + 1)
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    
    # 計算 S12 平均值
    s12_means = np.zeros(num_bins)
    s11_means = np.zeros(num_bins)
    mu_means = np.zeros(num_bins)
    counts = np.zeros(num_bins)
    
    for i in range(len(y)):
        bin_idx = np.searchsorted(bin_edges, y[i], side='right') - 1
        if 0 <= bin_idx < num_bins:
            s12_means[bin_idx] += s12[i]
            s11_means[bin_idx] += s11[i]
            if not np.isnan(mu[i]):
                mu_means[bin_idx] += mu[i]
            counts[bin_idx] += 1
    
    valid = counts > 0
    s12_means[valid] /= counts[valid]
    s11_means[valid] /= counts[valid]
    mu_means[valid] /= counts[valid]
    s12_means[~valid] = np.nan
    s11_means[~valid] = np.nan
    mu_means[~valid] = np.nan
    
    # 繪製 S12 平均值
    plt.figure(figsize=(8, 4))
    plt.plot(bin_centers, s12_means, 'o')
    plt.xlabel('Y Position')
    plt.ylabel('Average S12 (MPa)')
    plt.title(f'Y-direction Averaged Shear Stress S12 - Position {RUPTURE_POSITION}')
    plt.grid(True)
    plt.tight_layout()
    # plt.savefig(f'./{AVERAGE_FOLDER}/Shear-Stress-S12-{RUPTURE_POSITION}-Y-Averaged.pdf', dpi=300)
    plt.show()
    plt.close()
    
    # 繪製 S11 平均值
    plt.figure(figsize=(8, 4))
    plt.plot(bin_centers, s11_means, 'o')
    plt.xlabel('Y Position')
    plt.ylabel('Average S11 (MPa)')
    plt.title(f'Y-direction Averaged Normal Stress S11 - Position {RUPTURE_POSITION}')
    plt.grid(True)
    plt.tight_layout()
    # plt.savefig(f'./{AVERAGE_FOLDER}/Shear-Stress-S11-{RUPTURE_POSITION}-Y-Averaged.pdf', dpi=300)
    plt.show()
    plt.close()
    
    # 繪製 μ 平均值
    plt.figure(figsize=(8, 4))
    plt.plot(bin_centers, mu_means, 'o')
    plt.xlabel('Y Position')
    plt.ylabel('Average μ')
    plt.title(f'Y-direction Averaged Friction Coefficient μ - Position {RUPTURE_POSITION}')
    plt.grid(True)
    plt.tight_layout()
    # plt.savefig(f'./{AVERAGE_FOLDER}/Shear-Stress-Mu-{RUPTURE_POSITION}-Y-Averaged.pdf', dpi=300)
    plt.show()
    plt.close()
    
    # === 3. Z 中線分析 ===
    
    NUM_Y_BINS = 100
    INITIAL_TOL_RATIO_Z = 0.01
    MAX_TOL_RATIO_Z = 0.05
    MIN_REQUIRED_POINTS_Z = 50
    
    z_min, z_max = np.min(z), np.max(z)
    z_mid = 0.5 * (z_min + z_max)
    z_span = z_max - z_min
    tol_ratio_z = INITIAL_TOL_RATIO_Z
    
    # 尋找 Z 中線附近的點
    for _ in range(5):
        tol_z = tol_ratio_z * z_span
        mid_mask_z = np.abs(z - z_mid) <= tol_z
        if np.count_nonzero(mid_mask_z) >= MIN_REQUIRED_POINTS_Z or tol_ratio_z >= MAX_TOL_RATIO_Z:
            break
        tol_ratio_z = min(tol_ratio_z * 2, MAX_TOL_RATIO_Z)
    
    # Y 方向分箱
    y_edges_mid = np.linspace(y_min, y_max, NUM_Y_BINS + 1)
    y_centers_mid = 0.5 * (y_edges_mid[:-1] + y_edges_mid[1:])
    s12_midline_y = np.full(NUM_Y_BINS, np.nan)
    
    if np.count_nonzero(mid_mask_z) > 0:
        counts_y = np.zeros(NUM_Y_BINS, dtype=int)
        for yy, ss in zip(y[mid_mask_z], s12[mid_mask_z]):
            bi = np.searchsorted(y_edges_mid, yy, side='right') - 1
            if 0 <= bi < NUM_Y_BINS:
                if np.isnan(s12_midline_y[bi]):
                    s12_midline_y[bi] = 0.0
                s12_midline_y[bi] += ss
                counts_y[bi] += 1
        good_y = counts_y > 0
        s12_midline_y[good_y] /= counts_y[good_y]
    else:
        # Fallback：每個 Y-bin 取最接近 Z-mid 的點
        for bi in range(NUM_Y_BINS):
            in_bin = (y >= y_edges_mid[bi]) & (y < y_edges_mid[bi+1])
            if not np.any(in_bin):
                continue
            idx_local = np.argmin(np.abs(z[in_bin] - z_mid))
            s12_midline_y[bi] = s12[in_bin][idx_local]
    
    # 繪製 Z 中線的 S12 vs Y
    plt.figure(figsize=(8, 4))
    plt.plot(y_centers_mid, s12_midline_y, 'o-')
    plt.xlabel('Y Position')
    plt.ylabel('S12 at Z-midline (MPa)')
    plt.title(f'Midline (Z={z_mid:.3f}) Shear Stress S12 vs Y - Position {RUPTURE_POSITION} | tol≈{tol_ratio_z*100:.1f}% span')
    plt.grid(True)
    plt.tight_layout()
    # plt.savefig(f'./{MIDLINE_FOLDER}/Shear-Stress-{RUPTURE_POSITION}-MidlineZ-S12-vs-Y.pdf', dpi=300)
    plt.show()
    plt.close()
    
    print(f"  Completed analysis for position {RUPTURE_POSITION}")

print("\nAll analyses complete!")